In [1]:
# --- Imports
import hashlib, json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

# --- Config
@dataclass(frozen=True)
class Config:
    RAW: str = "fdv_and_mc_data_raw.csv"
    OUTDIR: str = "outputs"
    START: str = "2022-06-01"
    END: str = "2025-06-01"
    MIN_WEEKS: int = 5
    OUTLIER_Q: float = 0.99
    FDV_CAP: float = 1e12
    FDV_VS_MC_MULT: int = 100
    FDV_MISMATCH: float = 1e10
    MC_MISMATCH_FLOOR: float = 1e6
    CS_MAX_OBS: float = 0.95        # token-level cap on within-token max(%CS)
    CS_DROP: float = 0.10           # drop tokens with any Δ%CS < -10%
    MIN_AVG_MC: int = 5_000_000
    MIN_LATEST_FDV: int | None = None   # paper: None
    TOP_N: int | None = 1000            # paper: 1000
    FREQ: str | None = None             # None matches paper ("no period alignment")

CFG = Config()

# --- IO helpers
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

OUTDIR = Path(CFG.OUTDIR); OUTDIR.mkdir(parents=True, exist_ok=True)
REMOVED_DIR = OUTDIR / "removed_tokens"; REMOVED_DIR.mkdir(exist_ok=True)

In [2]:
def load_raw(cfg: Config = CFG) -> pd.DataFrame:
    df = pd.read_csv(cfg.RAW)
    df = df.rename(columns={"token_name":"symbol","fdv":"FDV","market_cap":"MC"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    # validity
    df = df[(df["FDV"] > 0) & (df["MC"] > 0)].copy()
    df["%CS"] = df["MC"] / df["FDV"]
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df[df["%CS"].between(0, 1, inclusive="right")]
    df = df.dropna(subset=["symbol","timestamp"])
    return df

In [3]:


def report_step(df_before: pd.DataFrame, df_after: pd.DataFrame, name: str) -> None:
    nb, na = len(df_before), len(df_after)
    tb = df_before["symbol"].nunique() if "symbol" in df_before.columns else 0
    ta = df_after["symbol"].nunique() if "symbol" in df_after.columns else 0
    print(f"{name}: {ta} tokens ({tb - ta} removed); {na} rows ({nb - na} removed)")

In [4]:
STEP_DESC = {
    "0_load":        "Load/rename/parse; compute %CS; basic positivity/finite checks",
    "1_window":      f"Date window [{CFG.START}, {CFG.END}] inclusive",
    "2_align":       f"Period alignment to {CFG.FREQ or 'None'}",
    "3_trim99":      f"Global trims at {CFG.OUTLIER_Q:.2%} for FDV and MC",
    "3p5_guards":    "FDV < 1e12; FDV ≤ 100×MC; NOT(FDV > 1e10 AND MC < 1e6)",
    "4_min_weeks":   f"Min observations per token ≥ {CFG.MIN_WEEKS}",
    "5_nonconst":    "Require within-token std(FDV)>0 and std(%CS)>0",
    "6_unlock_dyn":  "Require within-token max(%CS) > min(%CS)",
    "6p5_token_cap": f"Token-level max(%CS) < {CFG.CS_MAX_OBS:.0%}",
    "6p6_cs_drop":   f"Drop tokens with any Δ%CS < -{CFG.CS_DROP:.0%}",
    "7_avg_mc":      f"Average MC ≥ ${CFG.MIN_AVG_MC:,.0f}",
    "7b_latest_fdv": "Latest FDV not enforced" if CFG.MIN_LATEST_FDV is None else f"Latest FDV ≥ ${CFG.MIN_LATEST_FDV:,.0f}",
    "8_topN":        f"Keep Top-N by latest FDV: N={CFG.TOP_N}" if CFG.TOP_N else "Top-N not applied",
    "9_finalize":    "TGE proxy + day_from_TGE; final dedupe/sort/select",
}

def _write_removed(step_key: str, before_syms: set, after_syms: set) -> int:
    removed = sorted(before_syms - after_syms)
    (REMOVED_DIR / f"{step_key}.txt").write_text("\n".join(removed), encoding="utf-8")
    return len(removed)

def _align_period(df: pd.DataFrame, freq: str | None) -> pd.DataFrame:
    if not freq or str(freq).upper() in {"NONE","NA","NO"}:
        return df.copy()
    f = str(freq).upper()
    if f == "W-SUN":
        period_ts = df["timestamp"].dt.to_period("W-SUN").dt.to_timestamp("S")
    else:
        period_ts = df["timestamp"].dt.floor("D")
    return (df.assign(_period_ts=period_ts)
              .sort_values(["symbol","_period_ts","timestamp"])
              .drop_duplicates(subset=["symbol","_period_ts"], keep="last")
              .drop(columns=["timestamp"])
              .rename(columns={"_period_ts":"timestamp"}))

def run_pipeline_with_snapshots(cfg: Config = CFG):
    cascade = []
    snaps = {}

    # 0
    base = load_raw(cfg).dropna(subset=["symbol"]).copy()
    snaps["0_load"] = base.copy()
    cascade.append({"step":"0_load","tokens_after":base["symbol"].nunique(),"rows_after":len(base)})

    # 1
    before = base.copy()
    df1 = base[(base["timestamp"] >= cfg.START) & (base["timestamp"] <= cfg.END)].copy()
    _write_removed("step1_date_window", set(before["symbol"]), set(df1["symbol"]))
    snaps["1_window"] = df1.copy(); cascade.append({"step":"1_window","tokens_after":df1["symbol"].nunique(),"rows_after":len(df1)})

    # 2 (optional align; here None)
    before = df1.copy()
    df2 = _align_period(df1, cfg.FREQ)
    _write_removed("step2_period_align", set(before["symbol"]), set(df2["symbol"]))
    snaps["2_align"] = df2.copy(); cascade.append({"step":"2_align","tokens_after":df2["symbol"].nunique(),"rows_after":len(df2)})

    # 3 trims
    before = df2.copy()
    qfdv = before["FDV"].quantile(cfg.OUTLIER_Q); qmc = before["MC"].quantile(cfg.OUTLIER_Q)
    df3 = before[(before["FDV"] < qfdv) & (before["MC"] < qmc)].copy()
    _write_removed("step3_trim99", set(before["symbol"]), set(df3["symbol"]))
    snaps["3_trim99"] = df3.copy(); cascade.append({"step":"3_trim99","tokens_after":df3["symbol"].nunique(),"rows_after":len(df3)})

    # 3.5 guards
    before = df3.copy()
    df35 = before[(before["FDV"] < cfg.FDV_CAP)]
    df35 = df35[(df35["FDV"] <= df35["MC"] * cfg.FDV_VS_MC_MULT)]
    df35 = df35[~((df35["FDV"] > cfg.FDV_MISMATCH) & (df35["MC"] < cfg.MC_MISMATCH_FLOOR))].copy()
    _write_removed("step3p5_guards", set(before["symbol"]), set(df35["symbol"]))
    snaps["3p5_guards"] = df35.copy(); cascade.append({"step":"3p5_guards","tokens_after":df35["symbol"].nunique(),"rows_after":len(df35)})

    # 4 min span
    before = df35.copy()
    counts = before["symbol"].value_counts()
    keep = counts[counts >= cfg.MIN_WEEKS].index
    df4 = before[before["symbol"].isin(keep)].copy()
    _write_removed("step4_min_weeks", set(before["symbol"]), set(df4["symbol"]))
    snaps["4_min_weeks"] = df4.copy(); cascade.append({"step":"4_min_weeks","tokens_after":df4["symbol"].nunique(),"rows_after":len(df4)})

    # 5 non-const
    before = df4.copy()
    fdv_std = before.groupby("symbol")["FDV"].std()
    cs_std  = before.groupby("symbol")["%CS"].std()
    keep    = fdv_std[(fdv_std > 0) & (cs_std > 0)].index
    df5     = before[before["symbol"].isin(keep)].copy()
    _write_removed("step5_nonconst", set(before["symbol"]), set(df5["symbol"]))
    snaps["5_nonconst"] = df5.copy(); cascade.append({"step":"5_nonconst","tokens_after":df5["symbol"].nunique(),"rows_after":len(df5)})

    # 6 unlock dyn
    before = df5.copy()
    rng = before.groupby("symbol")["%CS"].agg(["min","max"])
    keep = rng[rng["max"] > rng["min"]].index
    df6  = before[before["symbol"].isin(keep)].copy()
    _write_removed("step6_unlock_dyn", set(before["symbol"]), set(df6["symbol"]))
    snaps["6_unlock_dyn"] = df6.copy(); cascade.append({"step":"6_unlock_dyn","tokens_after":df6["symbol"].nunique(),"rows_after":len(df6)})

    # 6.5 token cap
    before = df6.copy()
    cs_max = before.groupby("symbol")["%CS"].max()
    keep   = cs_max[cs_max < cfg.CS_MAX_OBS].index
    df65   = before[before["symbol"].isin(keep)].copy()
    _write_removed("step6p5_token_cs_cap", set(before["symbol"]), set(df65["symbol"]))
    snaps["6p5_token_cap"] = df65.copy(); cascade.append({"step":"6p5_token_cap","tokens_after":df65["symbol"].nunique(),"rows_after":len(df65)})

    # 6.6 abrupt drops
    before = df65.copy()
    dsort = before.sort_values(["symbol","timestamp"]).copy()
    dsort["cs_diff"] = dsort.groupby("symbol")["%CS"].diff()
    drop_syms = set(dsort.loc[dsort["cs_diff"] < -cfg.CS_DROP, "symbol"].unique())
    df66 = before[~before["symbol"].isin(drop_syms)].copy()
    (REMOVED_DIR / "step6p6_tokens_with_cs_drops_lt_threshold.txt").write_text("\n".join(sorted(drop_syms)), encoding="utf-8")
    _write_removed("step6p6_after_drop", set(before["symbol"]), set(df66["symbol"]))
    snaps["6p6_cs_drop"] = df66.copy(); cascade.append({"step":"6p6_cs_drop","tokens_after":df66["symbol"].nunique(),"rows_after":len(df66)})

    # 7 avg MC
    before = df66.copy()
    avg_mc = before.groupby("symbol")["MC"].mean()
    keep   = avg_mc[avg_mc >= cfg.MIN_AVG_MC].index
    df7    = before[before["symbol"].isin(keep)].copy()
    _write_removed("step7_min_avg_mc", set(before["symbol"]), set(df7["symbol"]))
    snaps["7_avg_mc"] = df7.copy(); cascade.append({"step":"7_avg_mc","tokens_after":df7["symbol"].nunique(),"rows_after":len(df7)})

    # 7b latest FDV (optional)
    before = df7.copy()
    if cfg.MIN_LATEST_FDV is not None:
        latest_date = before["timestamp"].max()
        latest_fdv  = before[before["timestamp"] == latest_date].groupby("symbol")["FDV"].mean()
        keep = latest_fdv[latest_fdv >= cfg.MIN_LATEST_FDV].index
        df7b = before[before["symbol"].isin(keep)].copy()
        _write_removed("step7b_latest_fdv", set(before["symbol"]), set(df7b["symbol"]))
    else:
        df7b = before.copy()
    snaps["7b_latest_fdv"] = df7b.copy(); cascade.append({"step":"7b_latest_fdv","tokens_after":df7b["symbol"].nunique(),"rows_after":len(df7b)})

    # 8 top-N by latest FDV (optional)
    before = df7b.copy()
    if cfg.TOP_N is not None:
        latest_date = before["timestamp"].max()
        latest_fdv_series = (before[before["timestamp"] == latest_date]
                             .groupby("symbol")["FDV"].mean()
                             .sort_values(ascending=False))
        top_syms = latest_fdv_series.head(int(cfg.TOP_N)).index
        df8 = before[before["symbol"].isin(top_syms)].copy()
        _write_removed("step8_topN_latest_fdv", set(before["symbol"]), set(df8["symbol"]))
    else:
        df8 = before.copy()
    snaps["8_topN"] = df8.copy(); cascade.append({"step":"8_topN","tokens_after":df8["symbol"].nunique(),"rows_after":len(df8)})

    # 9 finalise
    before = df8.copy()
    df9 = before.copy()
    df9["TGE_date"] = df9.groupby("symbol")["timestamp"].transform("min")
    df9["day_from_TGE"] = (df9["timestamp"] - df9["TGE_date"]).dt.days
    df9 = (df9.drop(columns=["TGE_date"])
             .drop_duplicates(subset=["timestamp","symbol"])
             .sort_values(["symbol","timestamp"])
             .reset_index(drop=True))
    df9 = df9[["timestamp","symbol","day_from_TGE","%CS","MC","FDV"]]
    snaps["9_finalize"] = df9.copy(); cascade.append({"step":"9_finalize","tokens_after":df9["symbol"].nunique(),"rows_after":len(df9)})

    # persist cascade + manifest
    cascade_df = pd.DataFrame(cascade)
    cascade_df.to_csv(OUTDIR / "filter_cascade_counts.csv", index=False)
    manifest = {
        "raw_path": str(CFG.RAW),
        "raw_sha256": sha256(Path(CFG.RAW)),
        "window": {"start": CFG.START, "end": CFG.END},
        "freq": CFG.FREQ,
        "params": {
            "outlier_q": CFG.OUTLIER_Q,
            "fdv_cap": CFG.FDV_CAP,
            "fdv_vs_mc_multiple": CFG.FDV_VS_MC_MULT,
            "fdv_mismatch_cap": CFG.FDV_MISMATCH,
            "mc_mismatch_floor": CFG.MC_MISMATCH_FLOOR,
            "cs_max_obs_token_level": CFG.CS_MAX_OBS,
            "cs_drop_threshold": CFG.CS_DROP,
            "min_weeks": CFG.MIN_WEEKS,
            "min_avg_mc": CFG.MIN_AVG_MC,
            "min_latest_fdv": CFG.MIN_LATEST_FDV,
            "top_n_latest_fdv": CFG.TOP_N,
        },
        "global_trim_fdv_q": float(qfdv),
        "global_trim_mc_q": float(qmc),
        "latest_date": str(df9["timestamp"].max().date()) if len(df9) else None,
        "final_rows": int(len(df9)),
        "final_tokens": int(df9["symbol"].nunique()) if len(df9) else 0,
    }
    (OUTDIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print("✓ Cascade:", OUTDIR / "filter_cascade_counts.csv")
    print("✓ Manifest:", OUTDIR / "run_manifest.json")
    print("✓ Token lists:", REMOVED_DIR.resolve())
    print(f"✅ Final dataset: {len(df9):,} rows; {df9['symbol'].nunique():,} tokens")
    return df9, cascade_df, snaps

In [5]:
def ensure_vars(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d = d[(d["MC"] > 0) & (d["FDV"] > 0)]
    d["cs_ratio"] = d["%CS"].astype(float)
    if "log_MC" not in d:  d["log_MC"]  = np.log(d["MC"])
    if "log_FDV" not in d: d["log_FDV"] = np.log(d["FDV"])
    return d

def fit_cluster_ols(y: str, d: pd.DataFrame) -> dict:
    X = sm.add_constant(d[["cs_ratio"]])
    cl = d["symbol"]
    res = sm.OLS(d[y], X).fit(cov_type="cluster", cov_kwds={"groups": cl})
    return {
        "coef": float(res.params.get("cs_ratio", np.nan)),
        "p": float(res.pvalues.get("cs_ratio", np.nan)),
        "R2": float(res.rsquared),
        "N": int(res.nobs),
    }

def order_steps(df: pd.DataFrame) -> pd.DataFrame:
    desired = [
        "0_load","1_window","2_align","3_trim99","3p5_guards",
        "4_min_weeks","5_nonconst","6_unlock_dyn","6p5_token_cap",
        "6p6_cs_drop","7_avg_mc","7b_latest_fdv","8_topN","9_finalize"
    ]
    omap = {k:i for i,k in enumerate(desired)}
    return (df.assign(_ord=df["step"].map(omap))
              .sort_values("_ord")
              .drop(columns=["_ord"]))

In [6]:
def save_corr_heatmaps(d: pd.DataFrame, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)
    dfh = d[(d["MC"] > 0) & (d["FDV"] > 0)].copy()
    dfh["log_MC"] = np.log(dfh["MC"]); dfh["log_FDV"] = np.log(dfh["FDV"])

    raw_corr = dfh[["%CS", "MC", "FDV"]].corr()
    plt.figure(figsize=(6,5))
    sns.heatmap(raw_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Correlation (Raw: %CS, MC, FDV)"); plt.tight_layout()
    plt.savefig(outdir / "corr_heatmap_raw.png", dpi=180); plt.close()
    raw_corr.to_csv(outdir / "corr_matrix_raw.csv", index=True)

    log_corr = dfh[["%CS", "log_MC", "log_FDV"]].corr()
    plt.figure(figsize=(6,5))
    sns.heatmap(log_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Correlation (Log: %CS, log(MC), log(FDV))"); plt.tight_layout()
    plt.savefig(outdir / "corr_heatmap_log.png", dpi=180); plt.close()
    log_corr.to_csv(outdir / "corr_matrix_log.csv", index=True)

def save_mini_dashboard(d: pd.DataFrame, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)
    plot_df = (d[(d["FDV"] > 0) & (d["MC"] > 0)]
               .replace([np.inf,-np.inf], np.nan)
               .dropna(subset=["FDV","MC","%CS"])
               .copy())

    fig, axs = plt.subplots(2, 2, figsize=(14,10))
    fig.suptitle("Variable Distributions", fontsize=16)

    # (1) FDV distribution (log10)
    axs[0,0].hist(np.log10(plot_df["FDV"].values), bins=80, edgecolor="none")
    axs[0,0].set_title("FDV Distribution (log10 scale)")
    axs[0,0].set_xlabel("log10(FDV)")
    axs[0,0].set_ylabel("Count")

    # (2) MC distribution (log10)
    axs[0,1].hist(np.log10(plot_df["MC"].values), bins=80, edgecolor="none")
    axs[0,1].set_title("Market Cap Distribution (log10 scale)")
    axs[0,1].set_xlabel("log10(MC)")
    axs[0,1].set_ylabel("Count")

    # (3) %CS distribution
    axs[1,0].hist(plot_df["%CS"].clip(0,1), bins=80, edgecolor="none")
    axs[1,0].set_title("%CS Distribution")
    axs[1,0].set_xlabel("Circulating Supply Ratio (%CS)")
    axs[1,0].set_ylabel("Count")

    # (4) FDV vs MC scatter (log-log)
    axs[1,1].scatter(plot_df["FDV"], plot_df["MC"], s=6, alpha=0.3)
    axs[1,1].set_xscale("log"); axs[1,1].set_yscale("log")
    axs[1,1].set_title("FDV vs Market Cap (log–log)")
    axs[1,1].set_xlabel("FDV")
    axs[1,1].set_ylabel("Market Cap")

    for ax in axs.flat:
        ax.grid(True, linestyle=":", lw=0.6, alpha=0.4)

    plt.tight_layout(rect=[0,0,1,0.95])
    fig.savefig(outdir / "variable_distributions.png", dpi=160)
    plt.close(fig)

In [7]:
def analyse_step_impacts(cfg: Config = CFG):
    final_df, cascade_df, snaps = run_pipeline_with_snapshots(cfg)
    impact_rows = []
    base_dir = Path(cfg.OUTDIR) / "step_impact"
    base_dir.mkdir(parents=True, exist_ok=True)

    for step_key, d in snaps.items():
        step_dir = base_dir / step_key
        step_dir.mkdir(exist_ok=True, parents=True)

        # regressions
        dat = ensure_vars(d)
        out_mc = fit_cluster_ols("log_MC", dat)
        out_fd = fit_cluster_ols("log_FDV", dat)

        impact_rows.append({
            "step": step_key,
            "tokens": int(d["symbol"].nunique()),
            "rows": int(len(d)),
            "coef_CS_logMC": out_mc["coef"], "p_CS_logMC": out_mc["p"], "R2_logMC": out_mc["R2"], "N_logMC": out_mc["N"],
            "coef_CS_logFDV": out_fd["coef"], "p_CS_logFDV": out_fd["p"], "R2_logFDV": out_fd["R2"], "N_logFDV": out_fd["N"],
            "clusters": int(d["symbol"].nunique()),
        })

        # artefacts
        save_corr_heatmaps(d, step_dir / "heatmaps")
        save_mini_dashboard(d, step_dir / "dashboard")

        # mini-manifest
        (step_dir / "manifest.json").write_text(json.dumps({
            "step": step_key,
            "tokens": int(d["symbol"].nunique()),
            "rows": int(len(d)),
            "start": str(d["timestamp"].min()) if len(d) else None,
            "end": str(d["timestamp"].max()) if len(d) else None,
        }, indent=2), encoding="utf-8")

    impact = order_steps(pd.DataFrame(impact_rows))
    impact_path = base_dir / "step_impact_summary.csv"
    impact.to_csv(impact_path, index=False)
    cascade_df.to_csv(base_dir / "cascade_counts.csv", index=False)

    print("✓ Impact summary ->", impact_path)
    print("✓ Cascade counts ->", base_dir / "cascade_counts.csv")
    print("Artifacts per step under ->", base_dir.resolve())
    return {"final_df": final_df, "cascade": cascade_df, "impact": impact, "snapshots": snaps}

In [8]:
def plot_step_overviews(impact: pd.DataFrame, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)
    d = order_steps(impact.copy())
    x = np.arange(len(d))

    # Coefficients
    fig, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(x, d["coef_CS_logMC"], marker="o", label="β(%CS) → log(MC)")
    ax1.plot(x, d["coef_CS_logFDV"], marker="o", label="β(%CS) → log(FDV)")
    ax1.axhline(0, linewidth=1, linestyle=":", color="gray")
    ax1.set_ylabel("Coefficient β")
    ax1.set_xticks(x); ax1.set_xticklabels(d["step"], rotation=35, ha="right")
    ax1.set_title("Coefficient path vs filtering steps")
    ax1.grid(True, linestyle=":", alpha=0.4)
    ax1.legend()
    fig.tight_layout()
    fig.savefig(outdir / "step_coefficients.png", dpi=180); plt.close(fig)

    # R²
    fig, ax = plt.subplots(figsize=(12,5))
    ax.plot(x, d["R2_logMC"], marker="o", label="R² log(MC)")
    ax.plot(x, d["R2_logFDV"], marker="o", label="R² log(FDV)")
    ax.set_ylabel("R²")
    ax.set_xticks(x); ax.set_xticklabels(d["step"], rotation=35, ha="right")
    ax.set_title("Model fit (R²) by filtering step")
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / "step_r2.png", dpi=180); plt.close(fig)

    # p-values (raw p)
    fig, ax = plt.subplots(figsize=(12,5))
    ax.plot(x, d["p_CS_logMC"], marker="o", label="p: %CS → log(MC)")
    ax.plot(x, d["p_CS_logFDV"], marker="o", label="p: %CS → log(FDV)")
    ax.axhline(0.05, linestyle="--", linewidth=1, color="red", label="p = 0.05")
    ax.set_ylabel("p-value")
    ax.set_ylim(0, 1)
    ax.set_xticks(x); ax.set_xticklabels(d["step"], rotation=35, ha="right")
    ax.set_title("Significance of %CS coefficient by filtering step")
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / "step_significance.png", dpi=180); plt.close(fig)

In [9]:
def analyse_step_impacts(cfg: Config = CFG):
    final_df, cascade_df, snaps = run_pipeline_with_snapshots(cfg)
    impact_rows = []
    base_dir = Path(cfg.OUTDIR) / "step_impact"
    base_dir.mkdir(parents=True, exist_ok=True)

    for step_key, d in snaps.items():
        step_dir = base_dir / step_key
        step_dir.mkdir(exist_ok=True, parents=True)

        # regressions
        dat = ensure_vars(d)
        out_mc = fit_cluster_ols("log_MC", dat)
        out_fd = fit_cluster_ols("log_FDV", dat)

        impact_rows.append({
            "step": step_key,
            "tokens": int(d["symbol"].nunique()),
            "rows": int(len(d)),
            "coef_CS_logMC": out_mc["coef"], "p_CS_logMC": out_mc["p"], "R2_logMC": out_mc["R2"], "N_logMC": out_mc["N"],
            "coef_CS_logFDV": out_fd["coef"], "p_CS_logFDV": out_fd["p"], "R2_logFDV": out_fd["R2"], "N_logFDV": out_fd["N"],
            "clusters": int(d["symbol"].nunique()),
        })

        # artefacts
        save_corr_heatmaps(d, step_dir / "heatmaps")
        save_mini_dashboard(d, step_dir / "dashboard")

        # mini-manifest
        (step_dir / "manifest.json").write_text(json.dumps({
            "step": step_key,
            "tokens": int(d["symbol"].nunique()),
            "rows": int(len(d)),
            "start": str(d["timestamp"].min()) if len(d) else None,
            "end": str(d["timestamp"].max()) if len(d) else None,
        }, indent=2), encoding="utf-8")

    impact = order_steps(pd.DataFrame(impact_rows))
    impact_path = base_dir / "step_impact_summary.csv"
    impact.to_csv(impact_path, index=False)
    cascade_df.to_csv(base_dir / "cascade_counts.csv", index=False)

    print("✓ Impact summary ->", impact_path)
    print("✓ Cascade counts ->", base_dir / "cascade_counts.csv")
    print("Artifacts per step under ->", base_dir.resolve())
    return {"final_df": final_df, "cascade": cascade_df, "impact": impact, "snapshots": snaps}

In [10]:
# Run the pipeline, write A (cascade/manifest), write B (step_impact/*), and plot summaries.

res = analyse_step_impacts(CFG)       # builds everything and returns tables
impact = res["impact"]                # tidy per-step regression summary (B)
final_df = res["final_df"]            # cleaned dataset (A)
print("\nTail of step-impact summary:")
display(impact)

# Optional overview plots for the report
plot_step_overviews(impact, Path(CFG.OUTDIR) / "step_impact")

# Optional: general dashboard on the final dataset
save_mini_dashboard(final_df, Path(CFG.OUTDIR) / "dashboards")

✓ Cascade: outputs/filter_cascade_counts.csv
✓ Manifest: outputs/run_manifest.json
✓ Token lists: /Users/katherinewebb/Cursor/Cedric's legacy/FDV:MC vs CS research/outputs/removed_tokens
✅ Final dataset: 66,253 rows; 826 tokens
✓ Impact summary -> outputs/step_impact/step_impact_summary.csv
✓ Cascade counts -> outputs/step_impact/cascade_counts.csv
Artifacts per step under -> /Users/katherinewebb/Cursor/Cedric's legacy/FDV:MC vs CS research/outputs/step_impact

Tail of step-impact summary:


,step,tokens,rows,coef_CS_logMC,p_CS_logMC,R2_logMC,N_logMC,coef_CS_logFDV,p_CS_logFDV,R2_logFDV,N_logFDV,clusters
0,0_load,3962,261567,1.349311,7.788076e-29,0.050345,261567,-1.711589,4.602599e-40,0.074861,261567,3962
1,1_window,3816,229626,1.544886,5.227972e-39,0.066402,229626,-1.484166,2.377780e-32,0.058463,229626,3816
2,2_align,3816,229626,1.544886,5.227972e-39,0.066402,229626,-1.484166,2.377780e-32,0.058463,229626,3816
3,3_trim99,3786,225708,1.483932,3.310004e-40,0.066179,225708,-1.336022,2.175382e-37,0.057178,225708,3786
4,3p5_guards,3744,221629,1.281384,1.195130e-31,0.050594,221629,-1.229889,4.024078e-31,0.048104,221629,3744
5,4_min_weeks,3500,221108,1.284421,1.305739e-31,0.050803,221108,-1.226502,8.704642e-31,0.047830,221108,3500
6,5_nonconst,2812,178438,1.394817,1.976630e-32,0.051884,178438,-1.356355,6.532612e-33,0.051069,178438,2812
7,6_unlock_dyn,2812,178438,1.394817,1.976630e-32,0.051884,178438,-1.356355,6.532612e-33,0.051069,178438,2812
8,6p5_token_cap,1867,138377,2.286983,1.285903e-35,0.083736,138377,-1.114796,6.085799e-10,0.021734,138377,1867
9,6p6_cs_drop,1598,114003,2.324694,2.949432e-29,0.085803,114003,-1.117323,3.994200e-08,0.021734,114003,1598


In [15]:
# --- Table 1: Pooled Regressions
def create_pooled_regressions_table(final_df: pd.DataFrame) -> pd.DataFrame:
    """Create pooled regressions table for log(MC) ~ %CS and log(FDV) ~ %CS"""
    d = ensure_vars(final_df)
    
    # Fit regressions
    X = sm.add_constant(d[["cs_ratio"]])
    cl = d["symbol"]
    
    # log(MC) ~ %CS
    mc_model = sm.OLS(d["log_MC"], X).fit(cov_type="cluster", cov_kwds={"groups": cl})
    
    # log(FDV) ~ %CS  
    fdv_model = sm.OLS(d["log_FDV"], X).fit(cov_type="cluster", cov_kwds={"groups": cl})
    
    # Create results table
    results = []
    
    # log(MC) results
    results.append({
        "Dependent Variable": "log(MC)",
        "β̂": f"{mc_model.params['cs_ratio']:.4f}",
        "SE (clustered)": f"{mc_model.bse['cs_ratio']:.4f}",
        "p-value": f"{mc_model.pvalues['cs_ratio']:.2e}",
        "R²": f"{mc_model.rsquared:.4f}",
        "N": f"{int(mc_model.nobs):,}"
    })
    
    # log(FDV) results
    results.append({
        "Dependent Variable": "log(FDV)",
        "β̂": f"{fdv_model.params['cs_ratio']:.4f}",
        "SE (clustered)": f"{fdv_model.bse['cs_ratio']:.4f}",
        "p-value": f"{fdv_model.pvalues['cs_ratio']:.2e}",
        "R²": f"{fdv_model.rsquared:.4f}",
        "N": f"{int(fdv_model.nobs):,}"
    })
    
    return pd.DataFrame(results)

# --- Table 2: Time-Windowed Regressions
def create_time_windowed_regressions_table(final_df: pd.DataFrame) -> pd.DataFrame:
    """Create time-windowed regressions table (Early: 0-150 days, Late: >150 days)"""
    d = ensure_vars(final_df)
    
    # Split into early and late periods
    early = d[d["day_from_TGE"] <= 150].copy()
    late = d[d["day_from_TGE"] > 150].copy()
    
    results = []
    
    for period_name, period_data in [("Early (0-150 days)", early), ("Late (>150 days)", late)]:
        if len(period_data) == 0:
            continue
            
        X = sm.add_constant(period_data[["cs_ratio"]])
        cl = period_data["symbol"]
        
        # log(MC) ~ %CS
        mc_model = sm.OLS(period_data["log_MC"], X).fit(cov_type="cluster", cov_kwds={"groups": cl})
        
        # log(FDV) ~ %CS
        fdv_model = sm.OLS(period_data["log_FDV"], X).fit(cov_type="cluster", cov_kwds={"groups": cl})
        
        # Add results
        results.append({
            "Period": period_name,
            "Dependent Variable": "log(MC)",
            "β̂": f"{mc_model.params['cs_ratio']:.4f}",
            "SE (clustered)": f"{mc_model.bse['cs_ratio']:.4f}",
            "p-value": f"{mc_model.pvalues['cs_ratio']:.2e}",
            "R²": f"{mc_model.rsquared:.4f}",
            "N": f"{int(mc_model.nobs):,}"
        })
        
        results.append({
            "Period": period_name,
            "Dependent Variable": "log(FDV)",
            "β̂": f"{fdv_model.params['cs_ratio']:.4f}",
            "SE (clustered)": f"{fdv_model.bse['cs_ratio']:.4f}",
            "p-value": f"{fdv_model.pvalues['cs_ratio']:.2e}",
            "R²": f"{fdv_model.rsquared:.4f}",
            "N": f"{int(fdv_model.nobs):,}"
        })
    
    return pd.DataFrame(results)

# --- Table 3: Sample Composition
def create_sample_composition_table(cascade_df: pd.DataFrame) -> pd.DataFrame:
    """Create sample composition table showing rows and tokens after each filter step"""
    
    # Define the key steps we want to highlight
    key_steps = [
        "1_window",      # Date window
        "2_align",
        "3_trim99", 
        "3p5_guards",     # Trims  
        "4_min_weeks",
        "5_nonconst",    # Non-const
        "6_unlock_dyn",  # Unlock dynamics
        "6p5_token_cap", # %CS cap (row-level)
        "7_avg_mc",      # MC ≥ $5m
        "7b_latest_fdv",
        "8_topN",         # Top 1000
        "9_finalize"
    ]


    # Filter to key steps and add step descriptions
    step_descriptions = {
        "1_window": "Date window",
        "2_align": f"Period alignment to {CFG.FREQ or 'None'}",
        "3_trim99": "Trims (99th percentile)",
        "3p5_guards":    "FDV < 1e12; FDV ≤ 100×MC; NOT(FDV > 1e10 AND MC < 1e6)",
        "4_min_weeks":   f"Min observations per token ≥ {CFG.MIN_WEEKS}",
        "5_nonconst": "Non-constant variables",
        "6_unlock_dyn": "Unlock dynamics",
        "6p5_token_cap": "%CS cap (row-level)",
        "7_avg_mc": "MC ≥ $5m",
        "7b_latest_fdv": "Latest FDV not enforced" if CFG.MIN_LATEST_FDV is None else f"Latest FDV ≥ ${CFG.MIN_LATEST_FDV:,.0f}",
        "8_topN": "Top 1000",
        "9_finalize":    "TGE proxy + day_from_TGE; final dedupe/sort/select",
    }
    
    composition = cascade_df[cascade_df["step"].isin(key_steps)].copy()
    composition["Filter Step"] = composition["step"].map(step_descriptions)
    composition["Tokens"] = composition["tokens_after"]
    composition["Rows"] = composition["rows_after"]
    
    # Select and reorder columns
    result = composition[["Filter Step", "Tokens", "Rows"]].copy()
    
    return result

# Generate all three tables
print("=== Table 1: Pooled Regressions ===")
pooled_table = create_pooled_regressions_table(final_df)
display(pooled_table)

print("\n=== Table 2: Time-Windowed Regressions ===")
time_windowed_table = create_time_windowed_regressions_table(final_df)
display(time_windowed_table)

print("\n=== Table 3: Sample Composition ===")
composition_table = create_sample_composition_table(res["cascade"])
display(composition_table)

# Save tables to CSV files
OUTDIR = Path(CFG.OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)

pooled_table.to_csv(OUTDIR / "table1_pooled_regressions.csv", index=False)
time_windowed_table.to_csv(OUTDIR / "table2_time_windowed_regressions.csv", index=False)
composition_table.to_csv(OUTDIR / "table3_sample_composition.csv", index=False)

print(f"\n✓ Tables saved to {OUTDIR}")
print("  - table1_pooled_regressions.csv")
print("  - table2_time_windowed_regressions.csv") 
print("  - table3_sample_composition.csv")


=== Table 1: Pooled Regressions ===


,Dependent Variable,β̂,SE (clustered),p-value,R²,N
0,log(MC),1.0337,0.1856,2.57e-08,0.0304,"66,253"
1,log(FDV),-2.1099,0.1952,3.10e-27,0.1134,"66,253"



=== Table 2: Time-Windowed Regressions ===


,Period,Dependent Variable,β̂,SE (clustered),p-value,R²,N
0,Early (0-150 days),log(MC),0.7956,0.1856,1.82e-05,0.0184,"16,938"
1,Early (0-150 days),log(FDV),-2.7242,0.1893,5.62e-47,0.1792,"16,938"
2,Late (>150 days),log(MC),1.2102,0.2163,2.21e-08,0.0387,"49,315"
3,Late (>150 days),log(FDV),-1.7395,0.2261,1.45e-14,0.0759,"49,315"



=== Table 3: Sample Composition ===


,Filter Step,Tokens,Rows
1,Date window,3816,229626
2,Period alignment to None,3816,229626
3,Trims (99th percentile),3786,225708
4,FDV < 1e12; FDV ≤ 100×MC; NOT(FDV > 1e10 AND M...,3744,221629
5,Min observations per token ≥ 5,3500,221108
6,Non-constant variables,2812,178438
7,Unlock dynamics,2812,178438
8,%CS cap (row-level),1867,138377
10,MC ≥ $5m,874,68770
11,Latest FDV not enforced,874,68770



✓ Tables saved to outputs
  - table1_pooled_regressions.csv
  - table2_time_windowed_regressions.csv
  - table3_sample_composition.csv


In [16]:
# Sensitivity grid: vary a couple of filters and re-fit headline log models
param_grid = [
    {"OUTLIER_Q": 0.99, "CS_MAX_OBS": 0.95, "MIN_AVG_MC": 5_000_000},
    {"OUTLIER_Q": 0.995,"CS_MAX_OBS": 0.95, "MIN_AVG_MC": 5_000_000},
    {"OUTLIER_Q": 0.99, "CS_MAX_OBS": 0.90, "MIN_AVG_MC": 5_000_000},
    {"OUTLIER_Q": 0.99, "CS_MAX_OBS": 0.95, "MIN_AVG_MC": 10_000_000},
]
rows = []
for p in param_grid:
    cfg = Config(**{**CFG.__dict__, **p})
    df_s, _, _ = run_pipeline_with_snapshots(cfg)
    d = df_s[(df_s["MC"]>0)&(df_s["FDV"]>0)].copy()
    d["cs_ratio"]=d["%CS"].astype(float); d["log_MC"]=np.log(d["MC"]); d["log_FDV"]=np.log(d["FDV"])
    X = sm.add_constant(d[["cs_ratio"]]); cl = d["symbol"]
    mc = sm.OLS(d["log_MC"], X).fit(cov_type="cluster", cov_kwds={"groups":cl})
    fd = sm.OLS(d["log_FDV"], X).fit(cov_type="cluster", cov_kwds={"groups":cl})
    rows.append({
        **p,
        "R2_logMC": mc.rsquared, "p_logMC": mc.pvalues["cs_ratio"], "beta_logMC": mc.params["cs_ratio"],
        "R2_logFDV": fd.rsquared, "p_logFDV": fd.pvalues["cs_ratio"], "beta_logFDV": fd.params["cs_ratio"],
        "tokens": d["symbol"].nunique(), "rows": len(d),
    })
sens = pd.DataFrame(rows).sort_values(list(param_grid[0].keys()))
sens.to_csv(Path(CFG.OUTDIR)/"regressions/sensitivity_grid.csv", index=False)

✓ Cascade: outputs/filter_cascade_counts.csv
✓ Manifest: outputs/run_manifest.json
✓ Token lists: /Users/katherinewebb/Cursor/Cedric's legacy/FDV:MC vs CS research/outputs/removed_tokens
✅ Final dataset: 66,253 rows; 826 tokens
✓ Cascade: outputs/filter_cascade_counts.csv
✓ Manifest: outputs/run_manifest.json
✓ Token lists: /Users/katherinewebb/Cursor/Cedric's legacy/FDV:MC vs CS research/outputs/removed_tokens
✅ Final dataset: 67,079 rows; 833 tokens
✓ Cascade: outputs/filter_cascade_counts.csv
✓ Manifest: outputs/run_manifest.json
✓ Token lists: /Users/katherinewebb/Cursor/Cedric's legacy/FDV:MC vs CS research/outputs/removed_tokens
✅ Final dataset: 62,199 rows; 783 tokens
✓ Cascade: outputs/filter_cascade_counts.csv
✓ Manifest: outputs/run_manifest.json
✓ Token lists: /Users/katherinewebb/Cursor/Cedric's legacy/FDV:MC vs CS research/outputs/removed_tokens
✅ Final dataset: 48,189 rows; 602 tokens
